<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>LoRA 적용 및 학습(LoRA Application & Training)</strong>에 대해 학습합니다.</br></br>
>ChatML 형식으로 데이터를 변환하고 SFTTrainer로 LoRA 학습을 학습해봅시다.

</br>

# ChatML 데이터 변환 (ChatML Conversion)
> 학습 데이터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ChatML 형식</mark>으로 변환하여 대화형 파인튜닝에 적합하게 만듭니다.

실제 LoRA를 적용할 때는 두 가지 핵심 결정이 필요합니다. 첫째, **Rank(r) 선택**입니다. rank가 높을수록 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">표현력이 올라가지만 메모리 사용량도 증가</mark>하며, 일반적으로 r=8~16이 균형점입니다. 둘째, **Target Modules 선택**으로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 레이어에 LoRA를 적용할지</mark> 결정합니다. Attention 레이어(`q_proj`, `v_proj`)만 적용하면 파라미터는 적고, MLP 레이어까지 포함하면 성능은 올라가지만 파라미터 수도 늘어납니다. 잘못된 설정은 학습 자원 낭비로 이어지므로, 목적에 맞는 설정 탐색이 중요합니다.

이 내용은 LoRA 원리(W' = W + BA 수식, Ch.5-1_001 참고)와 Hugging Face Transformers API, PEFT 라이브러리(`LoraConfig`, `get_peft_model`), SFT(정답 쌍으로 지도 학습하는 방식)에 대한 기본 이해를 바탕으로 합니다.

In [ ]:
# TODO 1: ChatML 변환 함수를 정의하여 system/user/assistant 메시지를 구성하고, 채팅 템플릿을 적용하여 ChatML 형식 텍스트를 반환한 뒤 데이터셋을 변환해봅시다.

def convert_to_chatml(example):
    """데이터를 ChatML 형식으로 변환"""
    messages = [
        {"role": "system", "content": "당신은 도움이 되는 AI 어시스턴트입니다."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]}
    ]
    # 토크나이저의 chat_template 적용
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# 데이터셋 변환
dataset = dataset.map(convert_to_chatml)
print(f"변환된 데이터 수: {len(dataset)}")
print(f"샘플 텍스트 (앞 100자):\n{dataset[0]['text'][:100]}...")

</br>

## ChatML 형식 예시

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">토큰</th>
      <th>역할</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">`<\</td><td>im_start\</td><td>>`</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">메시지 시작</mark></td></tr>
    <tr><td style="text-align:center">`<\</td><td>im_end\</td><td>>`</td><td>메시지 종료</td></tr>
    <tr><td style="text-align:center"><code>system/user/assistant</code></td><td>역할 구분</td></tr>
  </tbody>
</table>

</br>

# SFTTrainer (Supervised Fine-Tuning)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Hugging Face TRL 라이브러리</mark>의 SFT 전용 트레이너입니다.

In [ ]:
# TODO 2: 학습 설정을 구성하여 output_dir="./results", num_train_epochs=3, per_device_train_batch_size=4, learning_rate=2e-4, max_seq_length=512 등을 설정하고, 트레이너로 학습을 실행해봅시다.

from trl import SFTTrainer, SFTConfig

training_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_seq_length=512,
    dataset_text_field="text",   # ChatML 변환된 필드명
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print(f"학습 설정:")
print(f"  에폭: {training_config.num_train_epochs}")
print(f"  배치 크기: {training_config.per_device_train_batch_size}")
print(f"  학습률: {training_config.learning_rate}")
print(f"  최대 시퀀스: {training_config.max_seq_length}")

trainer.train()

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">학습 설정</th>
      <th style="text-align:center">값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">에폭</td><td style="text-align:center">3</td></tr>
    <tr><td style="text-align:center">배치 크기</td><td style="text-align:center">4</td></tr>
    <tr><td style="text-align:center">학습률</td><td style="text-align:center">0.0002</td></tr>
    <tr><td style="text-align:center">최대 시퀀스</td><td style="text-align:center">512</td></tr>
  </tbody>
</table>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">Step</th>
      <th style="text-align:center">Loss</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">10</td><td style="text-align:center">2.3456</td></tr>
    <tr><td style="text-align:center">20</td><td style="text-align:center">1.8765</td></tr>
    <tr><td style="text-align:center">30</td><td style="text-align:center">1.4532</td></tr>
    <tr><td style="text-align:center">40</td><td style="text-align:center">1.2345</td></tr>
    <tr><td style="text-align:center">50</td><td style="text-align:center">1.0987</td></tr>
  </tbody>
</table>

</br>

## 주요 학습 설정

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">권장값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>learning_rate</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습률</mark></td><td style="text-align:center">2e-4 ~ 5e-5</td></tr>
    <tr><td style="text-align:center"><code>num_train_epochs</code></td><td>에폭 수</td><td style="text-align:center">1~3</td></tr>
    <tr><td style="text-align:center"><code>per_device_train_batch_size</code></td><td>배치 크기</td><td style="text-align:center">2~8</td></tr>
    <tr><td style="text-align:center"><code>max_seq_length</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">최대 시퀀스 길이</mark></td><td style="text-align:center">512~2048</td></tr>
    <tr><td style="text-align:center"><code>dataset_text_field</code></td><td>텍스트 필드명</td><td style="text-align:center">"text"</td></tr>
  </tbody>
</table>

💡Unsloth 가속
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Unsloth</mark>는 LoRA 학습을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">2~5배 가속</mark>하는 라이브러리입니다.</br>
> `FastLanguageModel.from_pretrained()`로 모델을 로드하면 자동으로 최적화됩니다.

💡max_seq_length 설정
> 데이터의 최대 길이보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">짧으면 잘림</mark>이 발생합니다.</br>
> VRAM이 허용하는 범위에서 충분히 크게 설정하세요.

</br>

# Label Masking과 train_on_responses_only
> 학습 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모델의 응답(response) 부분만 loss를 계산</mark>하고, 지시문(instruction) 부분은 무시하는 기법입니다.

SFT(Supervised Fine-Tuning)에서 전체 시퀀스에 대해 loss를 계산하면, 모델은 **사용자의 질문까지 "외우려고"** 합니다. 이는 비효율적이며 과적합의 원인이 됩니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Label Masking</mark>은 instruction 부분의 label을 **-100**으로 설정하여 `CrossEntropyLoss`가 해당 토큰을 무시하게 만드는 기법입니다. 이를 통해 모델은 **"답변을 잘 생성하는 것"**에만 집중하여 학습합니다.

## Label Masking 동작 원리

<div style="text-align:center">

</div>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구분</th>
      <th style="text-align:center">Label 값</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Instruction 부분</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">-100</mark></td><td><code>CrossEntropyLoss(ignore_index=-100)</code>에 의해 loss 계산에서 제외</td></tr>
    <tr><td style="text-align:center">Response 부분</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제 토큰 ID</mark></td><td>모델이 예측해야 하는 정답 토큰 — loss 계산 대상</td></tr>
  </tbody>
</table>

## Unsloth의 train_on_responses_only

> Unsloth(및 TRL)에서는 `train_on_responses_only` 함수로 Label Masking을 간편하게 적용할 수 있습니다.

```python
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",   # instruction 시작 구분자
    response_part="<start_of_turn>model\n",      # response 시작 구분자
)
```

<mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">`instruction_part`</mark>부터 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">`response_part`</mark> 직전까지의 토큰이 **-100으로 마스킹**됩니다. 구분자 문자열은 사용하는 chat template에 따라 달라지며, Gemma-3의 경우 위와 같이 설정합니다.

In [ ]:
# TODO 4: Label Masking이 올바르게 적용되었는지 확인해봅시다.
# trainer의 첫 번째 배치에서 labels를 꺼내 -100(마스킹)과 실제 토큰 ID의 비율을 확인합니다.

sample = trainer.train_dataset[0]
input_ids = sample["input_ids"]
labels = sample["labels"]

# -100인 토큰 수 (마스킹된 instruction 부분)
masked_count = sum(1 for label in labels if label == -100)
# 실제 토큰 ID 수 (학습 대상인 response 부분)
unmasked_count = len(labels) - masked_count

print(f"전체 토큰 수: {len(labels)}")
print(f"마스킹된 토큰 (-100): {masked_count} ({masked_count/len(labels)*100:.1f}%)")
print(f"학습 대상 토큰: {unmasked_count} ({unmasked_count/len(labels)*100:.1f}%)")

# 마스킹 경계 시각화 (처음 20개 토큰)
print(f"\n처음 20개 토큰의 labels:")
for i, label in enumerate(labels[:20]):
    token = tokenizer.decode([input_ids[i]])
    status = "MASKED (-100)" if label == -100 else f"LEARN  ({label})"
    print(f"  [{i:3d}] {status}  →  '{token}'")

💡왜 -100인가?
> PyTorch의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">`nn.CrossEntropyLoss`</mark>는 기본적으로 `ignore_index=-100`이 설정되어 있습니다.</br>
> label이 -100인 위치는 loss 계산에서 자동으로 제외되므로, 별도의 mask 텐서 없이도 특정 토큰을 학습에서 빼낼 수 있습니다.

💡Label Masking 없이 학습하면?
> instruction 부분까지 loss에 포함되면 모델이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">"질문을 외우는 데"</mark> 파라미터 용량을 낭비합니다.</br>
> 특히 instruction이 긴 경우(few-shot 프롬프트 등) 비효율이 더욱 심해지며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">response-only 학습이 일반적으로 더 높은 성능</mark>을 보입니다.

</br>

# 학습 전후 비교 (Before vs After)
> 파인튜닝의 효과를 확인하기 위해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 입력에 대한 학습 전/후 응답</mark>을 비교합니다.

학습 전 베이스 모델의 응답을 미리 저장해 두고, 학습 후 동일 입력으로 추론하여 비교하면 파인튜닝 효과를 직관적으로 확인할 수 있습니다. 비교 시에는 반드시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습에 사용하지 않은 데이터</mark>로 테스트해야 공정한 평가가 가능합니다.

In [ ]:
# TODO 5: 학습 전후 비교를 위해, 학습 전 베이스 모델의 응답을 저장해봅시다.
# (이 셀은 학습 전에 실행해야 합니다)

# 테스트용 프롬프트 (학습 데이터에 없는 새로운 질문)
test_prompts = [
    {"role": "system", "content": "당신은 도움이 되는 AI 어시스턴트입니다."},
    {"role": "user", "content": "LoRA와 Full Fine-tuning의 차이점을 설명해줘."}
]

# 학습 전 응답 저장
input_text = tokenizer.apply_chat_template(
    test_prompts,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True
    )

before_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== 학습 전 베이스 모델 응답 ===")
print(before_response.strip())

In [ ]:
# TODO 6: 학습 후 동일한 프롬프트로 추론하여 학습 전 응답과 비교해봅시다.
# (이 셀은 학습 후에 실행해야 합니다)

# 학습 후 응답 생성 (동일 프롬프트 사용)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True
    )

after_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# 전후 비교 출력
print("=" * 60)
print("학습 전후 비교")
print("=" * 60)
print(f"\n[학습 전 (Before)]")
print(before_response.strip())
print(f"\n[학습 후 (After)]")
print(after_response.strip())
print("=" * 60)

💡학습 전후 비교 팁
> 비교 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">temperature=0.0</mark>으로 설정하면 deterministic한 응답을 얻을 수 있어 더 정확한 비교가 가능합니다.</br>
> 학습 전 응답을 변수에 저장해 두지 않으면 학습 후 비교가 불가능하므로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 전에 반드시 TODO 5를 먼저 실행</mark>하세요.

</br>

# LoRA 수학적 원리 (Low-Rank Adaptation)
> 원본 가중치 행렬 $W$를 직접 수정하지 않고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">저차원 행렬 $A$, $B$의 곱</mark>으로 업데이트를 근사합니다.

$$W' = W + \Delta W = W + BA$$

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">항목</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">$W \in \mathbb{R}^{d \times k}$</td><td>사전학습된 원본 가중치 (고정)</td></tr>
    <tr><td style="text-align:center">$A \in \mathbb{R}^{r \times k}$</td><td>저차원 행렬 A (학습)</td></tr>
    <tr><td style="text-align:center">$B \in \mathbb{R}^{d \times r}$</td><td>저차원 행렬 B (학습, 초기값 0)</td></tr>
    <tr><td style="text-align:center">$r$</td><td>rank — $r \ll \min(d, k)$</td></tr>
  </tbody>
</table>

</br>

## LoRA 핵심 하이퍼파라미터

In [ ]:
# TODO 3: LoraConfig를 사용하여 r=8, lora_alpha=16, target_modules(q_proj, k_proj, v_proj, o_proj)를 설정하고,
# get_peft_model로 모델에 LoRA를 적용한 뒤 학습 파라미터 수를 확인해봅시다.

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                       # rank: 저차원 크기 (클수록 표현력↑, 메모리↑)
    lora_alpha=16,             # 스케일링 값: alpha/r이 실제 스케일. 보통 r×2
    target_modules=[           # LoRA를 적용할 레이어 이름
        "q_proj", "k_proj",    # Attention Query, Key
        "v_proj", "o_proj",    # Attention Value, Output
    ],
    lora_dropout=0.05,         # 과적합 방지 드롭아웃
    bias="none",               # bias는 학습하지 않음
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# trainable params: 3,407,872 || all params: 6,742,609,920 || trainable%: 0.0506%

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">권장값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>r</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">rank</mark> — 저차원 크기</td><td style="text-align:center">8, 16, 32, 64</td></tr>
    <tr><td style="text-align:center"><code>lora_alpha</code></td><td>스케일링 값 (<code>alpha/r</code>이 적용됨)</td><td style="text-align:center"><code>r × 2</code></td></tr>
    <tr><td style="text-align:center"><code>target_modules</code></td><td>LoRA를 적용할 레이어</td><td style="text-align:center"><code>q_proj</code>, <code>v_proj</code> 등</td></tr>
    <tr><td style="text-align:center"><code>lora_dropout</code></td><td>드롭아웃 비율</td><td style="text-align:center">0.0 ~ 0.1</td></tr>
  </tbody>
</table>

💡lora_alpha 역할
> `lora_alpha`는 LoRA 출력에 곱해지는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">스케일 계수</mark>입니다.</br>
> 실제 적용 스케일 = `alpha / r` — `r`이 커질 때 alpha도 같이 올려 균형을 맞춥니다.

💡target_modules 선택 기준
> Attention 레이어(`q_proj`, `k_proj`, `v_proj`, `o_proj`)에 적용하는 것이 일반적으로 효과적입니다.</br>
> MLP 레이어(`gate_proj`, `up_proj`, `down_proj`)까지 포함하면 성능이 오르지만 파라미터 수도 증가합니다.